# 02 - Cargar datastore y mosaic dataset

Segunda etapa del flujo Geosupport. Este notebook usa el `04_ready_for_datastore.csv` generado en la etapa 1 para copiar las imagenes al datastore, agregarlas al mosaic dataset, construir footprints y actualizar campos criticos.

Ejecutar primero con `DRY_RUN = True`. Cuando la revision sea correcta, cambiar a `DRY_RUN = False`.

In [2]:
from datetime import datetime
from pathlib import Path
import importlib
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'core').exists():
    for candidate in [Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'core').exists():
            PROJECT_ROOT = candidate
            break

if not (PROJECT_ROOT / 'core').exists():
    raise FileNotFoundError('No se encontro el folder core. Ejecuta el notebook desde la raiz del proyecto o desde flujo_geosupport_etapas.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FLOW_DIR = PROJECT_ROOT / 'flujo_geosupport_etapas'

import core.mosaic_loader as mosaic_loader
mosaic_loader = importlib.reload(mosaic_loader)
from core.mosaic_loader import process_mosaic_load_row

print('Proyecto raiz:', PROJECT_ROOT)
print('Folder flujo:', FLOW_DIR)
print('Modulo carga:', mosaic_loader.__file__)

Proyecto raiz: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport
Folder flujo: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas
Modulo carga: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\core\mosaic_loader.py


## Parametros

La entrada normal es el CSV `04_ready_for_datastore.csv` de la etapa 1. Esta etapa genera sus propios archivos de resultado en una carpeta fija, reemplazando resultados anteriores.

In [3]:
READY_FOR_DATASTORE_CSV = FLOW_DIR / 'outputs' / 'etapa_01_preparar_paths_datastore' / '04_ready_for_datastore.csv'

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"
IMAGE_SERVICE_NAME = 'CL_MLP_PAO_IF_Ortho_Geosupport'

PROJECT_VALUE = 'PAO'
SENSOR_VALUE = 'DJI Mavic Enterprise'
MAXPS_VALUE = 10000
LOWPS_VALUE = 0.15
MINPS_VALUE = 0

# Seguridad operacional.
DRY_RUN = True
OVERWRITE_COPY = False
SKIP_EXISTING_MOSAIC_NAME = True

# Usar None para procesar todo. Para prueba controlada usar, por ejemplo, 1 o 5.
LIMIT_ROWS = None

run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = FLOW_DIR / 'outputs' / 'etapa_02_carga_datastore_mosaico'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('CSV entrada:', READY_FOR_DATASTORE_CSV)
print('Mosaic dataset:', PATH_MOSAIC_DATASET)
print('Salida:', OUTPUT_DIR)
print('DRY_RUN:', DRY_RUN)
print('OVERWRITE_COPY:', OVERWRITE_COPY)
print('SKIP_EXISTING_MOSAIC_NAME:', SKIP_EXISTING_MOSAIC_NAME)
print('LIMIT_ROWS:', LIMIT_ROWS)

CSV entrada: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_01_preparar_paths_datastore\04_ready_for_datastore.csv
Mosaic dataset: \\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport
Salida: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_02_carga_datastore_mosaico
DRY_RUN: True
OVERWRITE_COPY: False
SKIP_EXISTING_MOSAIC_NAME: True
LIMIT_ROWS: None


## 1. Leer manifiesto listo de etapa 1

Se procesan solo registros con `ready_for_datastore = True`. El CSV de entrada debe tener origen (`path`) y destino (`destination_path`).

In [4]:
ready_df = pd.read_csv(READY_FOR_DATASTORE_CSV)

required_columns = ['path', 'destination_path', 'expected_name', 'expected_file_name', 'expected_date_token', 'spatial_sector_raw']
missing_columns = [column for column in required_columns if column not in ready_df.columns]
if missing_columns:
    raise ValueError(f'Faltan columnas requeridas en {READY_FOR_DATASTORE_CSV}: {missing_columns}')

if 'ready_for_datastore' in ready_df.columns:
    ready_df = ready_df[ready_df['ready_for_datastore'].astype(str).str.lower().isin(['true', '1', 'yes'])].copy()

if LIMIT_ROWS is not None:
    ready_df = ready_df.head(int(LIMIT_ROWS)).copy()

print(f'Registros a procesar: {len(ready_df)}')
display(ready_df[['file_name', 'expected_file_name', 'destination_path', 'spatial_sector_raw']].head(20))

Registros a procesar: 33


,file_name,expected_file_name,destination_path,spatial_sector_raw
0,GEOSP-TRN-001426_GS_ORTOFOTO EB2 28-12-2025.tif,CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3
1,GEOSP-TRN-001492_GS_ORTOFOTO_Estacion de bombe...,CL_MLP_PAO_IF_Ortho_25_12_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3
2,GEOSP-TRN-001553_GS_ORTOFOTO_EB2_20-12-2025.tif,CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3
3,GEOSP-TRN-001642_ORTOFOTO_EB2_03-01-2026.tif,CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3
4,GEOSP-TRN-001732_GS_ORTOFOTO_EB2_17-01-26.tif,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3
5,GEOSP-TRN-001784_GS_ORTOFOTO_Estacion de bombe...,CL_MLP_PAO_IF_Ortho_26_01_23_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3
6,GEOSP-TRN-001868_GS_ORTOFOTO EB2 04-02-2026.tif,CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3
7,GEOSP-TRN-001975_GS_ORTOFOTO_Estacion de bombe...,CL_MLP_PAO_IF_Ortho_26_02_20_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3
8,GEOSP-TRN-002016_GS_ORTOFOTO_EB2_26-02-26.tif,CL_MLP_PAO_IF_Ortho_26_02_26_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3
9,GEOSP-TRN-002096_GS_ORTOFOTO_EB2_07-03-2026.tif,CL_MLP_PAO_IF_Ortho_26_03_07_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3


## 2. Preparar campos criticos del mosaic dataset

Los campos se derivan del manifest de etapa 1 y de valores fijos del proyecto. La URL usa la ruta relativa dentro del datastore.

In [5]:
def date_token_to_iso(date_token):
    if not date_token or pd.isna(date_token):
        return None
    parts = str(date_token).split('_')
    if len(parts) != 3:
        return None
    year, month, day = parts
    return f'20{year}-{month}-{day}'


def build_imageserver_url(row):
    if not row.get('destination_folder') or not row.get('destination_date_folder') or not row.get('expected_file_name'):
        return None
    relative_file_id = f".\\{row['destination_folder']}\\{row['destination_date_folder']}\\{row['expected_file_name']}"
    return f'https://sig.aminerals.cl/imgdyn/rest/services/CL_MLP_PAO/{IMAGE_SERVICE_NAME}/ImageServer/file?id={relative_file_id}&rasterId='


load_df = ready_df.copy()
load_df['Name'] = load_df['expected_name']
load_df['Raster'] = load_df['expected_file_name']
load_df['Path_Destino'] = load_df['destination_path']
load_df['Sector'] = load_df['spatial_sector_raw']
load_df['Fecha_Adqui'] = load_df['expected_date_token'].map(date_token_to_iso)
load_df['URL'] = load_df.apply(build_imageserver_url, axis=1)
load_df['Proyecto'] = PROJECT_VALUE
load_df['Sensor'] = SENSOR_VALUE
load_df['Fecha_Publ'] = datetime.now().strftime('%Y-%m-%d')
load_df['MaxPS'] = MAXPS_VALUE
load_df['LowPS'] = LOWPS_VALUE
load_df['MinPS'] = MINPS_VALUE
load_df['ProductName'] = 'OBJECTID del registro en el mosaic dataset'

critical_columns = ['Name', 'Raster', 'Path_Destino', 'Sector', 'Fecha_Adqui', 'URL', 'Proyecto', 'Sensor', 'Fecha_Publ', 'MaxPS', 'LowPS', 'MinPS', 'ProductName']
missing_critical = load_df[critical_columns].isna().sum().reset_index(name='null_count').rename(columns={'index': 'field'})

display(load_df[['file_name'] + critical_columns].head(20))
display(missing_critical)

,file_name,Name,Raster,Path_Destino,Sector,Fecha_Adqui,URL,Proyecto,Sensor,Fecha_Publ,MaxPS,LowPS,MinPS,ProductName
0,GEOSP-TRN-001426_GS_ORTOFOTO EB2 28-12-2025.tif,CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2025-12-28,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset
1,GEOSP-TRN-001492_GS_ORTOFOTO_Estacion de bombe...,CL_MLP_PAO_IF_Ortho_25_12_10_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_25_12_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2025-12-10,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset
2,GEOSP-TRN-001553_GS_ORTOFOTO_EB2_20-12-2025.tif,CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2025-12-20,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset
3,GEOSP-TRN-001642_ORTOFOTO_EB2_03-01-2026.tif,CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-01-03,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset
4,GEOSP-TRN-001732_GS_ORTOFOTO_EB2_17-01-26.tif,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-01-17,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset
5,GEOSP-TRN-001784_GS_ORTOFOTO_Estacion de bombe...,CL_MLP_PAO_IF_Ortho_26_01_23_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_01_23_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-01-23,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset
6,GEOSP-TRN-001868_GS_ORTOFOTO EB2 04-02-2026.tif,CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-02-04,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset
7,GEOSP-TRN-001975_GS_ORTOFOTO_Estacion de bombe...,CL_MLP_PAO_IF_Ortho_26_02_20_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_02_20_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-02-20,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset
8,GEOSP-TRN-002016_GS_ORTOFOTO_EB2_26-02-26.tif,CL_MLP_PAO_IF_Ortho_26_02_26_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_02_26_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-02-26,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset
9,GEOSP-TRN-002096_GS_ORTOFOTO_EB2_07-03-2026.tif,CL_MLP_PAO_IF_Ortho_26_03_07_MonteAranda-NSTC-...,CL_MLP_PAO_IF_Ortho_26_03_07_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,MonteAranda-NSTC-Km-84p2-a-82p3,2026-03-07,https://sig.aminerals.cl/imgdyn/rest/services/...,PAO,DJI Mavic Enterprise,2026-06-19,10000,0.15,0,OBJECTID del registro en el mosaic dataset


,field,null_count
0,Name,0
1,Raster,0
2,Path_Destino,0
3,Sector,0
4,Fecha_Adqui,0
5,URL,0
6,Proyecto,0
7,Sensor,0
8,Fecha_Publ,0
9,MaxPS,0


## 3. Ejecutar copia, carga al mosaico, footprint y atributos

Secuencia por cada imagen: copiar al datastore, agregar raster al mosaic dataset, construir footprint y actualizar atributos. Si `DRY_RUN = True`, no se escribe en el datastore ni en el mosaico.

In [6]:
results = []

for index, row in load_df.iterrows():
    print(f"{index + 1}/{len(load_df)} - {row['Name']}")
    result = process_mosaic_load_row(
        row,
        PATH_MOSAIC_DATASET,
        overwrite_copy=OVERWRITE_COPY,
        skip_existing_mosaic_name=SKIP_EXISTING_MOSAIC_NAME,
        maxps_value=MAXPS_VALUE,
        lowps_value=LOWPS_VALUE,
        minps_value=MINPS_VALUE,
        dry_run=DRY_RUN,
    )
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df.head(30))

for column in ['copy_status', 'mosaic_add_status', 'footprint_status', 'attribute_status', 'overall_status']:
    if column in results_df.columns:
        display(results_df[column].value_counts(dropna=False).reset_index(name='count').rename(columns={'index': column}))

1/33 - CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-Km-84p2-a-82p3
2/33 - CL_MLP_PAO_IF_Ortho_25_12_10_MonteAranda-NSTC-Km-84p2-a-82p3
3/33 - CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-Km-84p2-a-82p3
4/33 - CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-Km-84p2-a-82p3
5/33 - CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-Km-84p2-a-82p3
6/33 - CL_MLP_PAO_IF_Ortho_26_01_23_MonteAranda-NSTC-Km-84p2-a-82p3
7/33 - CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-Km-84p2-a-82p3
8/33 - CL_MLP_PAO_IF_Ortho_26_02_20_MonteAranda-NSTC-Km-84p2-a-82p3
9/33 - CL_MLP_PAO_IF_Ortho_26_02_26_MonteAranda-NSTC-Km-84p2-a-82p3
10/33 - CL_MLP_PAO_IF_Ortho_26_03_07_MonteAranda-NSTC-Km-84p2-a-82p3
11/33 - CL_MLP_PAO_IF_Ortho_26_03_13_MonteAranda-NSTC-Km-84p2-a-82p3
12/33 - CL_MLP_PAO_IF_Ortho_26_03_22_MonteAranda-NSTC-Km-84p2-a-82p3
13/33 - CL_MLP_PAO_IF_Ortho_26_03_27_MonteAranda-NSTC-Km-84p2-a-82p3
14/33 - CL_MLP_PAO_IF_Ortho_26_04_04_MonteAranda-NSTC-Km-84p2-a-82p3
15/33 - CL_MLP_PAO_IF_Ortho_26_04_12_MonteA

,file_name,Name,destination_path,overall_status,source_path,copy_status,copy_error,mosaic_add_status,mosaic_add_error,footprint_status,footprint_error,attribute_status,attribute_rows_updated,attribute_missing_fields,attribute_error
0,GEOSP-TRN-001426_GS_ORTOFOTO EB2 28-12-2025.tif,CL_MLP_PAO_IF_Ortho_25_12_28_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None
1,GEOSP-TRN-001492_GS_ORTOFOTO_Estacion de bombe...,CL_MLP_PAO_IF_Ortho_25_12_10_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None
2,GEOSP-TRN-001553_GS_ORTOFOTO_EB2_20-12-2025.tif,CL_MLP_PAO_IF_Ortho_25_12_20_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None
3,GEOSP-TRN-001642_ORTOFOTO_EB2_03-01-2026.tif,CL_MLP_PAO_IF_Ortho_26_01_03_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None
4,GEOSP-TRN-001732_GS_ORTOFOTO_EB2_17-01-26.tif,CL_MLP_PAO_IF_Ortho_26_01_17_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None
5,GEOSP-TRN-001784_GS_ORTOFOTO_Estacion de bombe...,CL_MLP_PAO_IF_Ortho_26_01_23_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None
6,GEOSP-TRN-001868_GS_ORTOFOTO EB2 04-02-2026.tif,CL_MLP_PAO_IF_Ortho_26_02_04_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None
7,GEOSP-TRN-001975_GS_ORTOFOTO_Estacion de bombe...,CL_MLP_PAO_IF_Ortho_26_02_20_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None
8,GEOSP-TRN-002016_GS_ORTOFOTO_EB2_26-02-26.tif,CL_MLP_PAO_IF_Ortho_26_02_26_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None
9,GEOSP-TRN-002096_GS_ORTOFOTO_EB2_07-03-2026.tif,CL_MLP_PAO_IF_Ortho_26_03_07_MonteAranda-NSTC-...,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Chacay_E...,dry_run,\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_D...,already_exists,None,dry_run,None,dry_run,None,dry_run,0,None,None


,copy_status,count
0,already_exists,32
1,dry_run,1


,mosaic_add_status,count
0,dry_run,33


,footprint_status,count
0,dry_run,33


,attribute_status,count
0,dry_run,33


,overall_status,count
0,dry_run,33


## 4. Exportar resultados de la etapa

Los resultados se escriben en una carpeta fija de la etapa y reemplazan los CSV anteriores.

In [8]:
summary_rows = [
    {'metric': 'run_timestamp', 'value': run_timestamp},
    {'metric': 'ready_for_datastore_csv', 'value': str(READY_FOR_DATASTORE_CSV)},
    {'metric': 'mosaic_dataset', 'value': PATH_MOSAIC_DATASET},
    {'metric': 'dry_run', 'value': DRY_RUN},
    {'metric': 'overwrite_copy', 'value': OVERWRITE_COPY},
    {'metric': 'skip_existing_mosaic_name', 'value': SKIP_EXISTING_MOSAIC_NAME},
    {'metric': 'maxps_value', 'value': MAXPS_VALUE},
    {'metric': 'lowps_value', 'value': LOWPS_VALUE},
    {'metric': 'minps_value', 'value': MINPS_VALUE},
    {'metric': 'product_name_value', 'value': 'OBJECTID del registro en el mosaic dataset'},
    {'metric': 'rows_to_process', 'value': len(load_df)},
]

for column in ['copy_status', 'mosaic_add_status', 'footprint_status', 'attribute_status', 'overall_status']:
    if column in results_df.columns:
        for status, count in results_df[column].value_counts(dropna=False).items():
            summary_rows.append({'metric': f'{column}_{status}', 'value': int(count)})

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / '00_summary.csv'
load_input_csv = OUTPUT_DIR / '01_load_input_with_attributes.csv'
load_results_csv = OUTPUT_DIR / '02_load_results.csv'
errors_csv = OUTPUT_DIR / '03_errors_review.csv'

summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
load_df.to_csv(load_input_csv, index=False, encoding='utf-8-sig')
results_df.to_csv(load_results_csv, index=False, encoding='utf-8-sig')

error_mask = pd.Series(False, index=results_df.index)
for column in ['copy_status', 'mosaic_add_status', 'footprint_status', 'attribute_status', 'overall_status']:
    if column in results_df.columns:
        error_mask = error_mask | results_df[column].astype(str).str.contains('error|failed|missing|not_found', case=False, na=False)
results_df[error_mask].to_csv(errors_csv, index=False, encoding='utf-8-sig')

display(summary_df)
print('Outputs exportados en:', OUTPUT_DIR)
print('Input enriquecido:', load_input_csv)
print('Resultados carga:', load_results_csv)
print('Revision errores:', errors_csv)

,metric,value
0,run_timestamp,20260619_085844
1,ready_for_datastore_csv,c:\Users\esrlrivero_adm\Documents\Geosupport\a...
2,mosaic_dataset,\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proye...
3,dry_run,True
4,overwrite_copy,False
5,skip_existing_mosaic_name,True
6,maxps_value,10000
7,lowps_value,0.15
8,minps_value,0
9,product_name_value,OBJECTID del registro en el mosaic dataset


Outputs exportados en: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_02_carga_datastore_mosaico
Input enriquecido: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_02_carga_datastore_mosaico\01_load_input_with_attributes.csv
Resultados carga: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_02_carga_datastore_mosaico\02_load_results.csv
Revision errores: c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\flujo_geosupport_etapas\outputs\etapa_02_carga_datastore_mosaico\03_errors_review.csv
